# Vision Colab T4 오프라인 임베딩

이 노트북은 Cloudflare Tunnel을 사용하지 않습니다. Google Drive로 동기화된 Vision 소스를 Colab T4에서 직접 Chunking·Embedding하고 결과를 같은 동기화 폴더의 `embedding-results`에 저장합니다.

처리 구조:

1. Drive의 Vision 소스 읽기
2. Backend와 동일한 파일 필터 및 1600/200 Chunking
3. T4 GPU의 Ollama `bge-m3:latest`로 1024차원 임베딩
4. 압축 JSONL Shard를 Drive에 원자적으로 저장
5. 중단 시 마지막 완료 Shard 다음부터 재개
6. 로컬 동기화 후 FastAPI가 결과 Package를 PostgreSQL/Qdrant로 Import


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Drive 연결과 T4 확인

Colab 메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택한 뒤 실행합니다.

In [3]:
from __future__ import annotations

import gzip
import hashlib
import json
import os
import shutil
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import requests
from google.colab import drive

drive.mount("/content/drive")

gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
    capture_output=True,
)
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise RuntimeError("Colab 런타임 유형을 T4 GPU로 변경한 뒤 다시 실행하세요.")

GPU_DESCRIPTION = gpu_check.stdout.strip()
print(f"GPU 준비 완료: {GPU_DESCRIPTION}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU 준비 완료: Tesla T4, 15360 MiB


## 2. 경로와 임베딩 계약 설정

`SYNC_ROOT`는 사용자가 확인한 실제 Google Drive 동기화 경로입니다. 결과는 그 아래 `embedding-results`에 저장됩니다.

In [4]:
SYNC_ROOT = Path(
    "/content/drive/Othercomputers/내 노트북/Documents/Vision"
)
SOURCE_ROOT = SYNC_ROOT
OUTPUT_ROOT = SYNC_ROOT / "embedding-results"
SOURCE_RELATIVE_PATH = "Vision"
USE_GIT_TRACKED_FILES = False

PROJECT_ID = "Vision"
SOURCE_ID = "colab-drive:Vision"
MODEL_NAME = "bge-m3:latest"
MODEL_ID = "bge-m3:latest"
INDEX_VERSION = "bge-m3-v1"
EXPECTED_DIMENSION = 1024
CHUNK_SIZE = 1600
CHUNK_OVERLAP = 200
MAX_INDEXABLE_FILE_BYTES = 16 * 1024 * 1024
EMBEDDING_BATCH_SIZE = 8
SHARD_RECORDS = 256
KEEP_ALIVE = "30m"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"

if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f"Vision 동기화 경로를 찾을 수 없습니다: {SOURCE_ROOT}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Source: {SOURCE_ROOT}")
print(f"Result: {OUTPUT_ROOT}")
print(f"Contract: {MODEL_ID} / {MODEL_NAME} / {EXPECTED_DIMENSION}D")


Source: /content/drive/Othercomputers/내 노트북/Documents/Vision
Result: /content/drive/Othercomputers/내 노트북/Documents/Vision/embedding-results
Contract: bge-m3:latest / bge-m3:latest / 1024D


## 3. Ollama 설치와 BGE-M3 GPU 서버 시작

Cloudflare는 설치하지 않습니다. Ollama는 Colab 내부 `127.0.0.1`에서만 실행됩니다.

In [5]:
if shutil.which("ollama") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
    subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=True,
    )


def ollama_ready() -> bool:
    try:
        return requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3).ok
    except requests.RequestException:
        return False


OLLAMA_LOG_PATH = Path("/content/vision-ollama.log")
if not ollama_ready():
    ollama_environment = os.environ.copy()
    ollama_environment.update(
        {
            "OLLAMA_HOST": "127.0.0.1:11434",
            "OLLAMA_KEEP_ALIVE": KEEP_ALIVE,
            "OLLAMA_LOAD_TIMEOUT": "10m",
            "OLLAMA_NUM_PARALLEL": "1",
            "OLLAMA_FLASH_ATTENTION": "1",
            "CUDA_VISIBLE_DEVICES": "0",
        }
    )
    ollama_log_handle = OLLAMA_LOG_PATH.open("w", encoding="utf-8")
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        env=ollama_environment,
        stdout=ollama_log_handle,
        stderr=subprocess.STDOUT,
    )
    for _ in range(90):
        if ollama_ready():
            break
        if ollama_process.poll() is not None:
            raise RuntimeError(OLLAMA_LOG_PATH.read_text(encoding="utf-8")[-4000:])
        time.sleep(1)
    else:
        raise TimeoutError("Ollama가 90초 안에 시작되지 않았습니다.")

subprocess.run(["ollama", "pull", MODEL_NAME], check=True)
print(subprocess.run(["ollama", "list"], text=True, capture_output=True, check=True).stdout)


NAME             ID              SIZE      MODIFIED               
bge-m3:latest    790764642607    1.2 GB    Less than a second ago    



## 4. Vision Backend와 동일한 파일·Chunk 규칙

파일 확장자, 제외 폴더, 문자 디코딩, Chunk 경계 및 Chunk ID 규칙을 Backend `repository_indexer.py`, `text.py`와 맞춥니다.

In [6]:
INDEXABLE_EXTENSIONS = {
    ".c", ".cc", ".cfg", ".conf", ".cpp", ".cs", ".css", ".cxx",
    ".dockerfile", ".env", ".go", ".h", ".hpp", ".htm", ".html",
    ".ini", ".java", ".js", ".json", ".jsx", ".kt", ".kts", ".md",
    ".mjs", ".php", ".properties", ".py", ".rb", ".rs", ".rst",
    ".scss", ".sh", ".sql", ".svelte", ".swift", ".toml", ".ts",
    ".tsx", ".txt", ".vue", ".xml", ".yaml", ".yml",
}
INDEXABLE_NAMES = {"dockerfile", "gemfile", "makefile", "procfile", "readme"}
EXCLUDED_INDEX_PARTS = {
    ".git", ".idea", ".next", ".venv", ".vscode", "__pycache__",
    "build", "coverage", "dist", "node_modules", "target", "vendor",
    "data", "embedding-results",
}
EXCLUDED_INDEX_SUFFIXES = {".lock", ".min.css", ".min.js", ".map"}
LANGUAGES = {
    ".c": "c", ".cc": "cpp", ".cpp": "cpp", ".cs": "csharp",
    ".css": "css", ".go": "go", ".h": "c", ".hpp": "cpp",
    ".html": "html", ".java": "java", ".js": "javascript",
    ".json": "json", ".jsx": "javascriptreact", ".kt": "kotlin",
    ".md": "markdown", ".php": "php", ".py": "python", ".rb": "ruby",
    ".rs": "rust", ".sh": "shellscript", ".sql": "sql", ".swift": "swift",
    ".toml": "toml", ".ts": "typescript", ".tsx": "typescriptreact",
    ".vue": "vue", ".xml": "xml", ".yaml": "yaml", ".yml": "yaml",
}


def safe_decode(data: bytes) -> tuple[str | None, str | None]:
    if b"\x00" in data:
        return None, None
    for encoding in ("utf-8-sig", "cp949"):
        try:
            return data.decode(encoding), encoding
        except UnicodeDecodeError:
            continue
    return None, None


def indexable_path(relative_path: str, size_bytes: int) -> bool:
    pure = PurePosixPath(relative_path)
    lower_name = pure.name.lower()
    if size_bytes <= 0 or size_bytes > MAX_INDEXABLE_FILE_BYTES:
        return False
    if any(part.lower() in EXCLUDED_INDEX_PARTS for part in pure.parts):
        return False
    if any(lower_name.endswith(suffix) for suffix in EXCLUDED_INDEX_SUFFIXES):
        return False
    return (
        pure.suffix.lower() in INDEXABLE_EXTENSIONS
        or lower_name in INDEXABLE_NAMES
        or lower_name.startswith("readme.")
    )


def chunk_text_with_metadata(text: str) -> list[dict[str, int | str]]:
    cleaned = text.replace("\r\n", "\n").strip()
    if not cleaned:
        return []
    if len(cleaned) <= CHUNK_SIZE:
        return [{
            "content": cleaned,
            "line_start": 1,
            "line_end": cleaned.count("\n") + 1,
        }]

    chunks = []
    start = 0
    while start < len(cleaned):
        hard_end = min(len(cleaned), start + CHUNK_SIZE)
        end = hard_end
        if hard_end < len(cleaned):
            boundary = max(
                cleaned.rfind("\n\n", start, hard_end),
                cleaned.rfind("\n", start, hard_end),
                cleaned.rfind(" ", start, hard_end),
            )
            if boundary > start + CHUNK_SIZE // 2:
                end = boundary
        chunk = cleaned[start:end].strip()
        if chunk:
            chunks.append({
                "content": chunk,
                "line_start": cleaned.count("\n", 0, start) + 1,
                "line_end": cleaned.count("\n", 0, end) + 1,
            })
        if end >= len(cleaned):
            break
        start = max(start + 1, end - min(CHUNK_OVERLAP, end - start - 1))
    return chunks


def atomic_write_json(path: Path, payload: dict) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary.replace(path)


## 5. 소스 Snapshot과 Chunk 목록 생성

Git 저장소라면 추적 중인 파일을 기준으로 읽습니다. Git이 없으면 제외 폴더를 건너뛰며 파일 시스템을 순회합니다.

In [7]:
def git_output(*arguments: str) -> str | None:
    result = subprocess.run(
        ["git", "-C", str(SOURCE_ROOT), *arguments],
        text=True,
        capture_output=True,
    )
    return result.stdout.strip() if result.returncode == 0 else None


revision = git_output("rev-parse", "HEAD")
git_files = git_output("ls-files", "-z")
dirty_output = git_output("status", "--porcelain")
git_dirty = bool(dirty_output) if dirty_output is not None else None

if USE_GIT_TRACKED_FILES and git_files is not None:
    relative_paths = sorted(path for path in git_files.split("\x00") if path)
    source_mode = "git_tracked_worktree"
else:
    relative_paths = sorted(
        path.relative_to(SOURCE_ROOT).as_posix()
        for path in SOURCE_ROOT.rglob("*")
        if path.is_file()
        and path.suffix.lower() != ".ipynb"
        and not any(part.lower() in EXCLUDED_INDEX_PARTS for part in path.relative_to(SOURCE_ROOT).parts)
    )
    source_mode = "filesystem"

manifest_rows = []
chunk_records = []
indexable_files = 0
total_bytes = 0

for relative_path in relative_paths:
    absolute_path = SOURCE_ROOT / relative_path
    if not absolute_path.is_file():
        continue
    raw = absolute_path.read_bytes()
    size_bytes = len(raw)
    total_bytes += size_bytes
    content_sha256 = hashlib.sha256(raw).hexdigest()
    manifest_rows.append({
        "relative_path": relative_path,
        "entry_type": "file",
        "size_bytes": size_bytes,
        "content_sha256": content_sha256,
    })

    if not indexable_path(relative_path, size_bytes):
        continue
    text, encoding = safe_decode(raw)
    if text is None or not text.strip():
        continue

    indexable_files += 1
    path_hash = hashlib.sha256(relative_path.encode("utf-8")).hexdigest()[:16]
    chunks = chunk_text_with_metadata(text)
    for ordinal, item in enumerate(chunks, start=1):
        chunk_records.append({
            "chunk_id": f"{path_hash}#chunk-{ordinal}",
            "document_id": path_hash,
            "path": relative_path,
            "language": LANGUAGES.get(PurePosixPath(relative_path).suffix.lower()),
            "content": str(item["content"]),
            "line_start": int(item["line_start"]),
            "line_end": int(item["line_end"]),
            "content_sha256": hashlib.sha256(str(item["content"]).encode("utf-8")).hexdigest(),
            "metadata": {"source_id": SOURCE_ID, "revision": revision, "encoding": encoding},
        })

manifest_encoded = json.dumps(
    manifest_rows,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")
manifest_sha256 = hashlib.sha256(manifest_encoded).hexdigest()
snapshot_id = f"snap_colab_{manifest_sha256[:20]}"
generation_seed = f"{PROJECT_ID}:{snapshot_id}:{MODEL_ID}:{INDEX_VERSION}"
generation_id = f"gen_colab_{hashlib.sha256(generation_seed.encode()).hexdigest()[:32]}"
artifact_id = f"{snapshot_id}_{MODEL_ID.replace(':', '_')}"
artifact_dir = OUTPUT_ROOT / PROJECT_ID / artifact_id
artifact_dir.mkdir(parents=True, exist_ok=True)

for record in chunk_records:
    record["metadata"]["snapshot_id"] = snapshot_id

print(json.dumps({
    "source_mode": source_mode,
    "revision": revision,
    "dirty": git_dirty,
    "files": len(manifest_rows),
    "indexable_files": indexable_files,
    "chunks": len(chunk_records),
    "total_bytes": total_bytes,
    "snapshot_id": snapshot_id,
    "generation_id": generation_id,
    "artifact_dir": str(artifact_dir),
}, ensure_ascii=False, indent=2))


{
  "source_mode": "filesystem",
  "revision": null,
  "dirty": null,
  "files": 9,
  "indexable_files": 8,
  "chunks": 11,
  "total_bytes": 7448,
  "snapshot_id": "snap_colab_eaa542ad66c20e15ce96",
  "generation_id": "gen_colab_d468319ee701889ef77993f298508380",
  "artifact_dir": "/content/drive/Othercomputers/내 노트북/Documents/Vision/embedding-results/Vision/snap_colab_eaa542ad66c20e15ce96_bge-m3_latest"
}


## 6. Artifact Manifest와 재개 Checkpoint 준비

동일 Snapshot·모델 결과가 이미 있으면 `progress.json`의 `next_chunk_index`부터 이어서 실행합니다.

In [8]:
manifest_path = artifact_dir / "manifest.json"
progress_path = artifact_dir / "progress.json"
complete_path = artifact_dir / "COMPLETE.json"

artifact_manifest = {
    "schema_version": "vision.embedding-artifact.v1",
    "artifact_id": artifact_id,
    "project_id": PROJECT_ID,
    "source_id": SOURCE_ID,
    "source_root": str(SOURCE_ROOT),
    "source_relative_path": SOURCE_RELATIVE_PATH,
    "source_mode": source_mode,
    "revision": revision,
    "git_dirty": git_dirty,
    "snapshot_id": snapshot_id,
    "generation_id": generation_id,
    "manifest_sha256": manifest_sha256,
    "model_id": MODEL_ID,
    "model_name": MODEL_NAME,
    "embedding_provider": "ollama-colab-t4",
    "embedding_dimension": EXPECTED_DIMENSION,
    "index_version": INDEX_VERSION,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "file_count": len(manifest_rows),
    "indexable_file_count": indexable_files,
    "chunk_count": len(chunk_records),
    "total_bytes": total_bytes,
    "created_at": datetime.now(timezone.utc).isoformat(),
}

if manifest_path.exists():
    existing_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    compatibility_fields = (
        "manifest_sha256", "model_id", "model_name", "embedding_dimension",
        "index_version", "chunk_size", "chunk_overlap", "chunk_count",
    )
    mismatches = [
        field for field in compatibility_fields
        if existing_manifest.get(field) != artifact_manifest.get(field)
    ]
    if mismatches:
        raise RuntimeError(f"기존 Artifact 계약과 다릅니다: {mismatches}")
else:
    atomic_write_json(manifest_path, artifact_manifest)

if progress_path.exists():
    progress = json.loads(progress_path.read_text(encoding="utf-8"))
else:
    progress = {
        "schema_version": "vision.embedding-progress.v1",
        "artifact_id": artifact_id,
        "next_chunk_index": 0,
        "embedded_chunks": 0,
        "shards": [],
        "updated_at": datetime.now(timezone.utc).isoformat(),
    }
    atomic_write_json(progress_path, progress)

next_chunk_index = int(progress.get("next_chunk_index", 0))
if not 0 <= next_chunk_index <= len(chunk_records):
    raise RuntimeError("progress.json의 next_chunk_index가 유효하지 않습니다.")

print(f"재개 위치: {next_chunk_index:,}/{len(chunk_records):,} chunks")


재개 위치: 11/11 chunks


## 7. T4 임베딩 실행 및 Shard 저장

각 Shard는 임시 파일에 완전히 쓴 뒤 이름을 교체합니다. 런타임이 중단되어도 완료된 Shard는 손상되지 않습니다.

In [9]:
def embed_batch(texts: list[str]) -> list[list[float]]:
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/embed",
        json={
            "model": MODEL_NAME,
            "input": texts,
            "truncate": False,
            "keep_alive": KEEP_ALIVE,
        },
        timeout=600,
    )
    response.raise_for_status()
    vectors = response.json().get("embeddings", [])
    if len(vectors) != len(texts):
        raise RuntimeError(f"요청 {len(texts)}, 응답 {len(vectors)}로 개수가 다릅니다.")
    if any(len(vector) != EXPECTED_DIMENSION for vector in vectors):
        raise RuntimeError("Embedding vector dimension이 1024가 아닙니다.")
    return vectors


def write_shard(records: list[dict], shard_number: int, start_index: int) -> dict:
    shard_name = f"part-{shard_number:05d}.jsonl.gz"
    shard_path = artifact_dir / shard_name
    temporary_path = artifact_dir / f"{shard_name}.tmp"
    with gzip.open(temporary_path, "wt", encoding="utf-8", compresslevel=6) as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")))
            handle.write("\n")
    temporary_path.replace(shard_path)
    checksum = hashlib.sha256(shard_path.read_bytes()).hexdigest()
    return {
        "name": shard_name,
        "sha256": checksum,
        "start_chunk_index": start_index,
        "end_chunk_index": start_index + len(records),
        "records": len(records),
        "size_bytes": shard_path.stat().st_size,
    }


pending_shard_records = []
pending_shard_start = next_chunk_index
shard_number = len(progress.get("shards", []))
current_index = next_chunk_index
started_at = time.monotonic()

while current_index < len(chunk_records):
    batch_end = min(current_index + EMBEDDING_BATCH_SIZE, len(chunk_records))
    batch = chunk_records[current_index:batch_end]
    vectors = embed_batch([record["content"] for record in batch])

    for record, vector in zip(batch, vectors):
        pending_shard_records.append({
            **record,
            "project_id": PROJECT_ID,
            "snapshot_id": snapshot_id,
            "generation_id": generation_id,
            "embedding_provider": "ollama-colab-t4",
            "embedding_model": MODEL_NAME,
            "embedding_model_id": MODEL_ID,
            "embedding_dimension": EXPECTED_DIMENSION,
            "index_version": INDEX_VERSION,
            "embedding": vector,
        })
    current_index = batch_end

    should_flush = (
        len(pending_shard_records) >= SHARD_RECORDS
        or current_index == len(chunk_records)
    )
    if should_flush:
        shard = write_shard(
            pending_shard_records,
            shard_number,
            pending_shard_start,
        )
        progress["shards"].append(shard)
        progress["next_chunk_index"] = current_index
        progress["embedded_chunks"] = current_index
        progress["updated_at"] = datetime.now(timezone.utc).isoformat()
        atomic_write_json(progress_path, progress)

        elapsed = max(time.monotonic() - started_at, 0.001)
        processed_now = current_index - next_chunk_index
        speed = processed_now / elapsed
        remaining = len(chunk_records) - current_index
        eta_minutes = remaining / speed / 60 if speed > 0 else 0
        print(
            f"{current_index:,}/{len(chunk_records):,} chunks "
            f"({current_index / max(len(chunk_records), 1) * 100:.1f}%) · "
            f"{speed:.2f} chunks/s · ETA {eta_minutes:.1f} min · {shard['name']}"
        )

        shard_number += 1
        pending_shard_records = []
        pending_shard_start = current_index

print("모든 Chunk 임베딩 완료")


모든 Chunk 임베딩 완료


## 8. 결과 검증과 완료 Marker

모든 Shard checksum, 레코드 수, 벡터 차원을 다시 확인한 뒤에만 `COMPLETE.json`을 생성합니다.

In [10]:
verified_records = 0
for shard in progress["shards"]:
    shard_path = artifact_dir / shard["name"]
    actual_checksum = hashlib.sha256(shard_path.read_bytes()).hexdigest()
    if actual_checksum != shard["sha256"]:
        raise RuntimeError(f"Shard checksum mismatch: {shard['name']}")
    shard_records = 0
    with gzip.open(shard_path, "rt", encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            if len(record.get("embedding", [])) != EXPECTED_DIMENSION:
                raise RuntimeError(f"Vector dimension mismatch: {shard['name']}")
            shard_records += 1
    if shard_records != shard["records"]:
        raise RuntimeError(f"Shard record count mismatch: {shard['name']}")
    verified_records += shard_records

if verified_records != len(chunk_records):
    raise RuntimeError(
        f"전체 Chunk 수 불일치: expected={len(chunk_records)}, actual={verified_records}"
    )

completion = {
    **artifact_manifest,
    "status": "completed",
    "verified_chunks": verified_records,
    "shards": len(progress["shards"]),
    "completed_at": datetime.now(timezone.utc).isoformat(),
}
atomic_write_json(complete_path, completion)

print(json.dumps({
    "status": "completed",
    "artifact_dir": str(artifact_dir),
    "chunks": verified_records,
    "shards": len(progress["shards"]),
    "model_id": MODEL_ID,
    "model_name": MODEL_NAME,
    "dimension": EXPECTED_DIMENSION,
}, ensure_ascii=False, indent=2))


{
  "status": "completed",
  "artifact_dir": "/content/drive/Othercomputers/내 노트북/Documents/Vision/embedding-results/Vision/snap_colab_eaa542ad66c20e15ce96_bge-m3_latest",
  "chunks": 11,
  "shards": 1,
  "model_id": "bge-m3:latest",
  "model_name": "bge-m3:latest",
  "dimension": 1024
}


## 9. 로컬 동기화 후 확인

완료 결과는 다음 구조로 로컬에 동기화됩니다.

```text
Vision/embedding-results/
└─ Vision/
   └─ snap_colab_..._bge-m3_latest/
      ├─ manifest.json
      ├─ progress.json
      ├─ part-00000.jsonl.gz
      ├─ part-00001.jsonl.gz
      └─ COMPLETE.json
```

`COMPLETE.json`이 존재해야 Import 가능한 완성 Package입니다. Cloudflare URL이나 포트 설정은 더 이상 필요하지 않습니다.

주의: Drive에 파일이 동기화되는 것과 Qdrant/PostgreSQL 반영은 별도 단계입니다. FastAPI Import 기능은 이 Package의 checksum과 모델 계약을 검증한 뒤 Qdrant와 PostgreSQL에 함께 반영해야 합니다.